# Voice STT/TTS Evaluation

Check speech-to-text and text-to-speech readiness.

Steps:
- Locate sample audio files.
- Attempt STT transcription if dependencies exist.
- Report TTS environment status.



In [ ]:
from __future__ import annotations

import json
import os
import sys
import subprocess
from pathlib import Path

# Resolve repo root from the notebook location.
REPO_ROOT = Path.cwd()
for parent in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (parent / 'scripts').exists() and (parent / 'notebooks').exists():
        REPO_ROOT = parent
        break

# Ensure local modules are importable.
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / 'src'))

PY = sys.executable

def run(cmd: list[str]) -> None:
    # Run a command from the repo root with PYTHONPATH set.
    env = os.environ.copy()
    env['PYTHONPATH'] = os.pathsep.join([str(REPO_ROOT / 'src'), str(REPO_ROOT)])
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=str(REPO_ROOT), check=True, env=env)

def show_json(rel_path: str) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        print(path.read_text(encoding='utf-8', errors='ignore')[:2000])
        return
    print(json.dumps(data, indent=2))

def list_dir(rel_path: str, limit: int = 20) -> None:
    path = REPO_ROOT / rel_path
    if not path.exists():
        print('Missing:', path)
        return
    print(f'\n{rel_path}/')
    for item in sorted(path.iterdir())[:limit]:
        print(' -', item.name)


In [ ]:
from pathlib import Path

summary = {
    'audio_samples': [],
    'stt': {},
    'tts_env': {},
}

voice_root = REPO_ROOT / 'data' / 'raw' / 'voice'
if voice_root.exists():
    audio_files = [p for p in voice_root.rglob('*.wav')]
    summary['audio_samples'] = [str(p.relative_to(REPO_ROOT)) for p in audio_files[:5]]
    print('Found audio files:', len(audio_files))
    for sample in summary['audio_samples']:
        print(' -', sample)
else:
    print('Missing voice root:', voice_root)


In [ ]:
# Attempt STT transcription if possible.
try:
    from app.services.stt.whisper_stt import WhisperSTT
except Exception as exc:
    WhisperSTT = None
    summary['stt']['error'] = str(exc)
    print('STT import error:', exc)

if WhisperSTT and summary['audio_samples']:
    sample_path = REPO_ROOT / summary['audio_samples'][0]
    try:
        result = WhisperSTT.transcribe_file(str(sample_path))
        summary['stt']['text'] = result.get('text')
        summary['stt']['language'] = result.get('language')
        summary['stt']['segments'] = result.get('segments', [])[:3]
        print('STT text preview:', summary['stt']['text'])
    except Exception as exc:
        summary['stt']['error'] = str(exc)
        print('STT failed:', exc)


In [ ]:
# Check TTS environment variables.
summary['tts_env'] = {
    'PIPER_BIN': bool(os.getenv('PIPER_BIN')),
    'PIPER_MODEL': bool(os.getenv('PIPER_MODEL')),
    'PIPER_CONFIG': bool(os.getenv('PIPER_CONFIG')),
}
print('TTS env:', summary['tts_env'])


In [ ]:
# Persist summary for quick reference.
report_dir = REPO_ROOT / 'reports'
report_dir.mkdir(parents=True, exist_ok=True)
summary_path = report_dir / 'evaluation_voice_stt_tts_summary.json'
summary_path.write_text(json.dumps(summary, indent=2))
print('Saved summary to', summary_path)


In [ ]:
# Quick artifact index for verification.
for folder in ['models', 'experiments', 'artifacts', 'runs', 'reports', 'logs']:
    path = REPO_ROOT / folder
    if not path.exists():
        continue
    print(f'\n{folder}/')
    for item in sorted(path.iterdir())[:20]:
        print(' -', item.name)
